# 3 · What a run supplies

A node is a function, so a helper is another function. What only the host has (who runs the graph, a clock, a client) is the run's context: one object of a class the host names on the registry, `NodeRegistry(context_type=AppContext)`. A node takes it as its first parameter, typed `Context[AppContext]`, and the host passes it as `graph.run(context=AppContext(...))`.

In [ ]:
from dataclasses import dataclass
from typing import Annotated

from conductor import Address, Context, Graph, GraphRecord, InvalidCallError, NodeRecord, NodeRegistry, Param, RecordConstant, RecordEdges, node
from conductor.widgets import Textarea

## A node with a helper

In [ ]:
def _words(text: str) -> list[str]:
    return text.split()


@node("word-count", title="Word count", description="Counts the words in a text", category="text")
def word_count(text: Annotated[str, Param(title="Text", widget=Textarea())]) -> Annotated[int, Param(title="Words")]:
    return len(_words(text))


word_count("the quick brown fox")

## What the run supplies

`AppContext` is the host's class. The node's first parameter has no widget, no edge and no value in the record.

In [ ]:
@dataclass(frozen=True)
class AppContext:
    """What the host gives every run: who is running the graph."""

    caller_name: str


@node("signed", title="Signed", description="Signs a text with the name of whoever ran the graph", category="text")
def signed(
    ctx: Context[AppContext],
    text: Annotated[str, Param(title="Text", widget=Textarea())],
) -> Annotated[str, Param(title="Signed")]:
    return f"{text} — {ctx.caller_name}"


registry = NodeRegistry(context_type=AppContext)
registry.add(word_count, signed)

## A need, not an input

`signed.context` names it. The inputs and the check a call goes through do not, and a test calls the node with the context first.

In [ ]:
print("inputs: ", list(signed.inputs))
print("context:", signed.context)
print(signed(AppContext(caller_name="Ida"), "Done"))

## Run it

In [ ]:
graph = Graph.compile(
    GraphRecord(
        nodes={
            "note": NodeRecord(node="signed", inputs={"text": RecordConstant(constant="The quick brown fox jumps over the lazy dog")}),
            "count": NodeRecord(node=word_count.id, inputs={"text": RecordEdges(sources=(Address("note", "result"),))}),
        },
        outputs={"words": Address("count", "result")},
    ), registry,
)

results = graph.run(context=AppContext(caller_name="Ida")).results
print(results["note"]["result"])
print(results["count"]["result"])

Without it, the call itself raises:

In [ ]:
try:
    graph.run()
except InvalidCallError as error:
    print(error)